In [11]:
import torch
from torch import nn


## Attention Block

In [19]:
class Attention_Block(nn.Module):
    def __init__(self,features,num_heads):
        super().__init__()
        self.num_heads=num_heads
        self.head_features=features//num_heads
        self.q=nn.Linear(features,features)
        self.v=nn.Linear(features,features)
        self.k=nn.Linear(features,features)
    def forward(self,x):
        query=self.q(x)
        value=self.v(x)
        key=self.k(x)
        query=query.view(query.shape[0],query.shape[1],self.num_heads,self.head_features)
        key=key.view(key.shape[0],key.shape[1],self.num_heads,self.head_features)
        value=value.view(value.shape[0],value.shape[1],self.num_heads,self.head_features)
        score=key@query.transpose(-2,-1)
        scaled_score=score/(self.head_features**0.5)
        scaled_score=torch.softmax(scaled_score,dim=-1)
        attention=scaled_score@value
        attention=attention.transpose(1,2)
        attention=attention.contiguous().view(x.shape[0],x.shape[1],-1)
        return attention

### Res-Net

In [20]:
x=torch.rand((2,4,8),dtype=torch.float32)
features=8
num_heads=4
block=Attention_Block(features,num_heads)
attention=block(x)
residue=attention+x
normalisation=nn.LayerNorm(features)
residue=normalisation(residue)
print(residue)

tensor([[[-6.2812e-01, -4.4134e-01,  8.9325e-01,  1.0700e+00,  2.4264e-01,
          -2.0564e+00, -1.5488e-01,  1.0748e+00],
         [ 1.0311e+00,  6.5955e-01, -1.4380e+00,  1.0741e+00, -1.3642e+00,
          -9.5877e-01,  4.6028e-01,  5.3598e-01],
         [ 1.0372e-01, -8.8735e-01,  2.4230e+00, -2.8552e-01, -1.1047e+00,
          -8.2882e-02, -2.3447e-02, -1.4278e-01],
         [ 7.9708e-02,  8.2161e-01, -4.9251e-01,  6.0477e-01, -2.6766e-01,
          -2.2696e+00,  3.5915e-01,  1.1645e+00]],

        [[ 6.1999e-01,  1.5209e-01, -2.4596e+00, -2.9274e-01,  1.0152e+00,
           4.8380e-01,  4.3411e-01,  4.7151e-02],
         [ 1.0747e+00, -1.8608e+00,  1.2224e+00,  4.8934e-01, -4.2328e-01,
          -1.0724e+00,  5.3516e-03,  5.6473e-01],
         [-1.4520e+00, -8.8426e-01,  2.4714e-01, -3.0476e-01,  1.1335e+00,
          -1.0263e+00,  1.1179e+00,  1.1688e+00],
         [-1.2915e+00,  1.4288e-03,  8.4623e-01,  1.0137e+00, -9.1111e-01,
          -1.4410e+00,  1.1061e+00,  6.7611e-01]